# 04 - Structured Agent Output

## Learning objective

Require the agent to return a validated Pydantic object containing an answer and a list of source URLs.

```text
User request -> agent and Tavily -> Pydantic validation -> AgentResponse
```

This notebook breaks the implementation into small, inspectable steps.

> **Quota note:** The final agent cell can make multiple Gemini and Tavily requests.


## 1. Check the active Python kernel

The executable should point to the Conda environment used for notebooks, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

If the course packages are not installed in the active kernel, remove the `#` and run the cell once. Then use **Kernel -> Restart Kernel**.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL and environment variables

`truststore` uses the Windows certificate store. The existing `.env` is loaded from the sibling script folder without printing secret values.


In [2]:
import os
from pathlib import Path

import truststore
from dotenv import load_dotenv

truststore.inject_into_ssl()

ENV_PATH = Path("../langchain-course/.env").resolve()
load_dotenv(dotenv_path=ENV_PATH)

print(".env exists:", ENV_PATH.exists())
print("Google key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))


.env exists: True
Google key loaded: True
Tavily key loaded: True
LangSmith key loaded: True
LangSmith tracing: true


## 4. Import Pydantic and agent components

`BaseModel` defines a data contract. `Field` supplies descriptions and defaults that help validation and model guidance.


In [3]:
from typing import List

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

print("Imports completed successfully.")


Imports completed successfully.


## 5. Define the nested `Source` schema

Every source must contain a string called `url`.


In [4]:
class Source(BaseModel):
    """Schema for one source used by the agent."""

    url: str = Field(
        description="The URL of the source"
    )


print(Source.model_json_schema())


{'description': 'Schema for one source used by the agent.', 'properties': {'url': {'description': 'The URL of the source', 'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'Source', 'type': 'object'}


## 6. Define the complete response schema

`List[Source]` means the field must be a list whose items follow the `Source` schema. `default_factory=list` creates a fresh empty list when no sources are supplied.


In [5]:
class AgentResponse(BaseModel):
    """Schema for an agent answer and its sources."""

    answer: str = Field(
        description="The agent's answer to the query"
    )

    sources: List[Source] = Field(
        default_factory=list,
        description="List of sources used to generate the answer",
    )


AgentResponse.model_json_schema()


{'$defs': {'Source': {'description': 'Schema for one source used by the agent.',
   'properties': {'url': {'description': 'The URL of the source',
     'title': 'Url',
     'type': 'string'}},
   'required': ['url'],
   'title': 'Source',
   'type': 'object'}},
 'description': 'Schema for an agent answer and its sources.',
 'properties': {'answer': {'description': "The agent's answer to the query",
   'title': 'Answer',
   'type': 'string'},
  'sources': {'description': 'List of sources used to generate the answer',
   'items': {'$ref': '#/$defs/Source'},
   'title': 'Sources',
   'type': 'array'}},
 'required': ['answer'],
 'title': 'AgentResponse',
 'type': 'object'}

## 7. Test Pydantic locally

This does not call any API. It demonstrates validation, attribute access, and serialization.


In [6]:
example_response = AgentResponse(
    answer="Example answer",
    sources=[
        Source(url="https://example.com/job-1"),
        Source(url="https://example.com/job-2"),
    ],
)

print(example_response.answer)
print(example_response.sources[0].url)
print(example_response.model_dump())


Example answer
https://example.com/job-1
{'answer': 'Example answer', 'sources': [{'url': 'https://example.com/job-1'}, {'url': 'https://example.com/job-2'}]}


## 8. Create Gemini, the Tavily tool, and the structured agent

The key addition is `response_format=AgentResponse`.


In [7]:
MODEL =  "gemini-3.5-flash-lite" #"gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    max_retries=2,
)

tools = [TavilySearch()]

agent = create_agent(
    model=llm,
    tools=tools,
    response_format=AgentResponse,
)

print("Agent created with response schema:", AgentResponse.__name__)


Agent created with response schema: AgentResponse


## 9. Build the request


In [8]:
question = (
    "Search for 3 job postings for an AI engineer "
    "using LangChain in the Bay Area on LinkedIn "
    "and list their details."
)

agent_input = {
    "messages": [HumanMessage(content=question)]
}

print(agent_input)


{'messages': [HumanMessage(content='Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.', additional_kwargs={}, response_metadata={})]}


## 10. Run the structured agent

> **This cell can make multiple Gemini and Tavily requests.**


In [9]:
result = agent.invoke(agent_input)
print("Agent execution completed.")


Agent execution completed.


## 11. Inspect the complete state and structured response

The final validated object is stored under `structured_response`.


In [10]:
print("Result keys:", result.keys())
print("Message count:", len(result["messages"]))
print("Structured response type:", type(result["structured_response"]))


Result keys: dict_keys(['messages', 'structured_response'])
Message count: 6
Structured response type: <class '__main__.AgentResponse'>


In [11]:
structured_response = result["structured_response"]

print("Answer:")
print(structured_response.answer)

print("Sources:")
for source in structured_response.sources:
    print(source.url)


Answer:
Here are 3 job postings related to AI Engineers using LangChain or working with LLMs and agents in the San Francisco Bay Area on LinkedIn:
Sources:
https://www.linkedin.com/jobs/view/deployed-engineer-bay-area-at-langchain-4392168401
https://www.linkedin.com/jobs/view/senior-ai-engineer-%E2%80%93-llm-rag-at-brightai-4281883251
https://www.linkedin.com/jobs/view/ai-llm-engineer-voice-%2B-vision-ai-at-hostbuddy-inc-4287800164


## 12. Convert the validated object for downstream use

A dictionary or JSON representation is convenient for APIs, databases, files, and other application components.


In [15]:
response_dictionary = structured_response.model_dump()
response_json = structured_response.model_dump_json(indent=2)

print(response_dictionary)
print("\n#################################\n")
print(response_json)


{'answer': 'Here are 3 job postings related to AI Engineers using LangChain or working with LLMs and agents in the San Francisco Bay Area on LinkedIn:', 'sources': [{'url': 'https://www.linkedin.com/jobs/view/deployed-engineer-bay-area-at-langchain-4392168401'}, {'url': 'https://www.linkedin.com/jobs/view/senior-ai-engineer-%E2%80%93-llm-rag-at-brightai-4281883251'}, {'url': 'https://www.linkedin.com/jobs/view/ai-llm-engineer-voice-%2B-vision-ai-at-hostbuddy-inc-4287800164'}]}

#################################

{
  "answer": "Here are 3 job postings related to AI Engineers using LangChain or working with LLMs and agents in the San Francisco Bay Area on LinkedIn:",
  "sources": [
    {
      "url": "https://www.linkedin.com/jobs/view/deployed-engineer-bay-area-at-langchain-4392168401"
    },
    {
      "url": "https://www.linkedin.com/jobs/view/senior-ai-engineer-%E2%80%93-llm-rag-at-brightai-4281883251"
    },
    {
      "url": "https://www.linkedin.com/jobs/view/ai-llm-engineer-voi

## 13. Why structured output matters

Free-form text is optimized for people. Structured output is optimized for software.

```text
Free text -> harder to parse reliably
Pydantic object -> predictable fields, types, and validation
```

## Practice exercises

- Add a `title` field to `Source`.
- Add `company`, `location`, and `job_url` fields to a `JobPosting` model.
- Make `sources` mandatory by removing `default_factory`.
- Save `model_dump()` to JSON after a successful API run.
